# Training Patch Classifiers from Per-Class Patch Folders

Click to open in: [[GitHub](https://github.com/TissueImageAnalytics/tiatoolbox/blob/develop/examples/12-training-patch-folder-classification.ipynb)][[Colab](https://colab.research.google.com/github/TissueImageAnalytics/tiatoolbox/blob/develop/examples/12-training-patch-folder-classification.ipynb)]


## About this notebook

This Jupyter notebook can run on a local Python environment or on Google Colab. If you run it on Colab, use the installation cell below and restart the runtime once installation finishes.


## About this demo

This example demonstrates model training for **case 1** in the training roadmap: per-class patch folders where each subfolder name defines the target class.

You will:

1. Create a small folder-structured patch dataset.
1. Build a `PatchFolderClassificationDataset`.
1. Train a classifier with `Trainer` and `ClassificationTask`.
1. Inspect generated checkpoints and resume training from `last.ckpt`.


## Optional: Colab setup

Skip this section when running in an environment where `tiatoolbox` is already installed.


In [ ]:
%%bash
apt-get -y install libopenjp2-7-dev libopenjp2-tools openslide-tools libpixman-1-dev | tail -n 1
pip install git+https://github.com/TissueImageAnalytics/tiatoolbox.git@develop | tail -n 1
echo "Installation is done."


If you are using Colab and ran the installation cell for the first time, restart the runtime before continuing.


## Imports and configuration


In [ ]:
from collections import defaultdict
from pathlib import Path
import shutil

import matplotlib.pyplot as plt
import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader, Subset

from tiatoolbox.models.training import (
    ClassificationTask,
    OptimizerConfig,
    PatchFolderClassificationDataset,
    SchedulerConfig,
    Trainer,
    TrainerConfig,
    create_optimizer,
    create_scheduler,
)
from tiatoolbox.utils.misc import imwrite

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")


## Create a synthetic per-class patch dataset

For a self-contained example, we generate synthetic patches and save them as PNG files using the expected folder structure.


In [ ]:
dataset_root = Path("tmp/patch_folder_training_demo")
if dataset_root.exists():
    shutil.rmtree(dataset_root)
dataset_root.mkdir(parents=True, exist_ok=True)

rng = np.random.default_rng(7)
class_specs = {
    "tumor": np.array([175, 70, 120], dtype=np.float32),
    "stroma": np.array([120, 165, 95], dtype=np.float32),
    "necrosis": np.array([200, 200, 120], dtype=np.float32),
}
patch_size = 64
samples_per_class = 60

for class_name, base_color in class_specs.items():
    class_dir = dataset_root / class_name
    class_dir.mkdir(parents=True, exist_ok=True)

    for sample_idx in range(samples_per_class):
        patch = np.ones((patch_size, patch_size, 3), dtype=np.float32) * base_color
        patch += rng.normal(0, 18, size=patch.shape)

        if class_name == "tumor":
            patch[:, ::4, 0] += 35
            patch[::4, :, 2] += 20
        elif class_name == "stroma":
            patch[:, ::6, 1] += 35
            patch[::6, :, 1] += 35
        else:
            yy, xx = np.ogrid[:patch_size, :patch_size]
            center = patch_size // 2
            radius = patch_size // 3
            core = (xx - center) ** 2 + (yy - center) ** 2 < radius**2
            patch[core] += np.array([30, 30, -20], dtype=np.float32)

        patch = np.clip(patch, 0, 255).astype(np.uint8)
        imwrite(class_dir / f"{class_name}_{sample_idx:03d}.png", patch)

print(f"Dataset path: {dataset_root.resolve()}")
for class_dir in sorted(dataset_root.iterdir()):
    patch_count = len(list(class_dir.glob("*.png")))
    print(f"{class_dir.name}: {patch_count} patches")


In [ ]:
fig, axes = plt.subplots(len(class_specs), 4, figsize=(10, 7))
for row_index, class_name in enumerate(class_specs):
    sample_paths = sorted((dataset_root / class_name).glob("*.png"))[:4]
    for col_index, sample_path in enumerate(sample_paths):
        image = plt.imread(sample_path)
        axes[row_index, col_index].imshow(image)
        axes[row_index, col_index].axis("off")
    axes[row_index, 0].set_ylabel(class_name, rotation=90, size=11)

plt.tight_layout()


## Build dataset and dataloaders


In [ ]:
dataset = PatchFolderClassificationDataset(root_dir=dataset_root)
print(f"Total patches: {len(dataset)}")
print("Class mapping:", dataset.class_to_idx)

def stratified_split_indices(samples, val_fraction=0.2, seed=123):
    rng = np.random.default_rng(seed)
    grouped = defaultdict(list)
    for item_index, (_, class_index) in enumerate(samples):
        grouped[class_index].append(item_index)

    train_indices = []
    val_indices = []
    for indices in grouped.values():
        indices = np.array(indices)
        rng.shuffle(indices)
        n_val = max(1, int(len(indices) * val_fraction))
        val_indices.extend(indices[:n_val].tolist())
        train_indices.extend(indices[n_val:].tolist())

    return train_indices, val_indices

train_indices, val_indices = stratified_split_indices(
    dataset.samples,
    val_fraction=0.2,
    seed=5,
)

train_dataset = Subset(dataset, train_indices)
val_dataset = Subset(dataset, val_indices)

print(f"Train patches: {len(train_dataset)}")
print(f"Val patches:   {len(val_dataset)}")


In [ ]:
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False, num_workers=0)

batch = next(iter(train_loader))
print("Image batch shape:", tuple(batch["image"].shape))
print("Target batch shape:", tuple(batch["target"].shape))
print("Image dtype:", batch["image"].dtype)
print("Target dtype:", batch["target"].dtype)


## Define model and trainer

The training loop is task-agnostic and only requires a PyTorch model that returns logits.


In [ ]:
num_classes = len(dataset.class_to_idx)

def build_model(num_output_classes):
    return nn.Sequential(
        nn.Conv2d(3, 16, kernel_size=3, padding=1),
        nn.ReLU(),
        nn.MaxPool2d(2),
        nn.Conv2d(16, 32, kernel_size=3, padding=1),
        nn.ReLU(),
        nn.AdaptiveAvgPool2d(1),
        nn.Flatten(),
        nn.Linear(32, num_output_classes),
    )

model = build_model(num_classes)

optimizer = create_optimizer(
    model,
    OptimizerConfig(name="adamw", lr=1e-3, weight_decay=1e-4),
)
scheduler = create_scheduler(
    optimizer,
    SchedulerConfig(name="cosine", t_max=8, eta_min=1e-5),
)

output_dir = Path("tmp/patch_folder_training_run")
output_dir.mkdir(parents=True, exist_ok=True)

trainer = Trainer(
    model=model,
    task=ClassificationTask(loss="cross_entropy"),
    optimizer=optimizer,
    scheduler=scheduler,
    train_loader=train_loader,
    val_loader=val_loader,
    config=TrainerConfig(
        max_epochs=8,
        device=device,
        amp=True,
        seed=42,
        output_dir=output_dir,
        monitor="val_loss",
        monitor_mode="min",
        log_every_n_steps=0,
    ),
)


## Train the model


In [ ]:
history = trainer.fit()
print(f"Finished epochs: {len(history)}")
print(f"Best epoch: {trainer.best_epoch}")
history[-1]


In [ ]:
epochs = [int(record["epoch"]) for record in history]
train_loss = [record["train_loss"] for record in history]
val_loss = [record.get("val_loss", np.nan) for record in history]
train_acc = [record["train_accuracy"] for record in history]
val_acc = [record.get("val_accuracy", np.nan) for record in history]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(epochs, train_loss, marker="o", label="train_loss")
axes[0].plot(epochs, val_loss, marker="o", label="val_loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].set_title("Loss")
axes[0].legend()

axes[1].plot(epochs, train_acc, marker="o", label="train_accuracy")
axes[1].plot(epochs, val_acc, marker="o", label="val_accuracy")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy")
axes[1].set_title("Accuracy")
axes[1].legend()

plt.tight_layout()


## Checkpoints and resume

`Trainer` writes `last.ckpt` (full state) and `best_model_weights.pth` (weights only).


In [ ]:
print("Saved checkpoint artifacts:")
for path in sorted(output_dir.glob("*")):
    print("-", path.name)

resumed_model = build_model(num_classes)
resumed_optimizer = create_optimizer(
    resumed_model,
    OptimizerConfig(name="adamw", lr=1e-3, weight_decay=1e-4),
)
resumed_scheduler = create_scheduler(
    resumed_optimizer,
    SchedulerConfig(name="cosine", t_max=10, eta_min=1e-5),
)

resumed_trainer = Trainer(
    model=resumed_model,
    task=ClassificationTask(loss="cross_entropy"),
    optimizer=resumed_optimizer,
    scheduler=resumed_scheduler,
    train_loader=train_loader,
    val_loader=val_loader,
    config=TrainerConfig(
        max_epochs=10,
        device=device,
        amp=True,
        seed=42,
        output_dir=output_dir,
        monitor="val_loss",
        monitor_mode="min",
        log_every_n_steps=0,
    ),
)

resumed_history = resumed_trainer.fit(resume_from=output_dir / "last.ckpt")
print(f"Resumed to epoch: {int(resumed_history[-1]['epoch'])}")


## Adapting this notebook to real data

Replace `dataset_root` with your own patch directory:

```text
my_patch_dataset/
  class_a/
    patch_0001.png
    patch_0002.png
  class_b/
    patch_0003.png
```

Then keep the same `PatchFolderClassificationDataset`, dataloaders, and `Trainer` workflow.
